In [2]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=fd51730d718e5a983b5095100cf66ad0c3bbc7c296a28286497d1df14f736dc8
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [2]:
import os
import csv
import re
import sys
import spacy
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from langdetect import detect
import pickle
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.tag import pos_tag
from nltk.probability import FreqDist

import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

In [1]:
import sklearn
import numpy
import scipy

print("sklearn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)


sklearn: 1.6.1
numpy: 2.0.2
scipy: 1.16.3


In [3]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("names")
nltk.download("wordnet")
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('tagsets')
nltk.download('tagsets_json')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Unzipping corpora/names.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package tagsets to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets.zip.
[nltk_data] Downloading package tagsets_json to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets_json.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_

True

In [4]:
STOPWORDS = nltk.corpus.stopwords.words("english")
NAMES = nltk.corpus.names.words()

In [72]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


True

In [6]:
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    "/content/drive/MyDrive/NLP1/Phishing.csv",
    engine="python"
)

In [7]:
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [8]:
df.drop(columns="Unnamed: 0",inplace=True)

In [9]:
df.columns

Index(['Email Text', 'Email Type'], dtype='object')

In [10]:
df['Email Text'].astype(str)

,Email Text
0,"re : 6 . 1100 , disc : uniformitarianism , re ..."
1,the other side of * galicismos * * galicismo *...
2,re : equistar deal tickets are you still avail...
3,\nHello I am your hot lil horny toy.\n I am...
4,software at incredibly low prices ( 86 % lower...
...,...
18645,date a lonely housewife always wanted to date ...
18646,request submitted : access request for anita ....
18647,"re : important - prc mtg hi dorn & john , as y..."
18648,press clippings - letter on californian utilit...


In [11]:
df['Email Text'].loc[3]

'\nHello I am your hot lil horny toy.\n    I am the one you dream About,\n    I am a very open minded person,\n    Love to talk about and any subject.\n    Fantasy is my way of life, \n    Ultimate in sex play.     Ummmmmmmmmmmmmm\n     I am Wet and ready for you.     It is not your looks but your imagination that matters most,\n     With My sexy voice I can make your dream come true...\n  \n     Hurry Up! call me let me Cummmmm for you..........................\nTOLL-FREE:             1-877-451-TEEN (1-877-451-8336)For phone billing:     1-900-993-2582\n-- \n_______________________________________________\nSign-up for your own FREE Personalized E-mail at Mail.com\nhttp://www.mail.com/?sr=signup'

In [12]:
df['Email Type'].value_counts(normalize=True)

,proportion
Email Type,
Safe Email,0.607078
Phishing Email,0.392922


In [13]:
df.dropna(inplace=True)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18634 entries, 0 to 18649
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Email Text  18634 non-null  object
 1   Email Type  18634 non-null  object
dtypes: object(2)
memory usage: 436.7+ KB


In [15]:
def get_language(x):
  try:
    return detect(x)
  except:
    return "unknown"

In [16]:
df["language"]=df['Email Text'].apply(get_language)
df['language'].value_counts()

,count
language,
en,17760
fr,580
es,53
ca,43
de,36
pt,32
nl,18
tr,12
it,11


In [17]:
df=df[df['language']=='en']

In [18]:
def clean_text(text: str) -> str:

    # Quitar HTML primero
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")

    # Lowercase
    text = text.lower()

    # Reemplazar URLs primero
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' url_token ',
        text
    )

    # Reemplazar emails
    text = re.sub(
        r'\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b',
        ' email_token ',
        text
    )

    # Remover números
    text = re.sub(r'\b\d+(\.\d+)?\b', ' ', text)

    # Remover caracteres no alfabéticos (pero conservar underscore)
    text = re.sub(r'[^a-z_\s]', ' ', text)

    # Normalizar espacios
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [19]:
df["clean_text"] = df["Email Text"].apply(clean_text)


In [20]:
df['Email Text'].iloc[3]


'\nHello I am your hot lil horny toy.\n    I am the one you dream About,\n    I am a very open minded person,\n    Love to talk about and any subject.\n    Fantasy is my way of life, \n    Ultimate in sex play.     Ummmmmmmmmmmmmm\n     I am Wet and ready for you.     It is not your looks but your imagination that matters most,\n     With My sexy voice I can make your dream come true...\n  \n     Hurry Up! call me let me Cummmmm for you..........................\nTOLL-FREE:             1-877-451-TEEN (1-877-451-8336)For phone billing:     1-900-993-2582\n-- \n_______________________________________________\nSign-up for your own FREE Personalized E-mail at Mail.com\nhttp://www.mail.com/?sr=signup'

In [21]:
df['clean_text'].iloc[3]


'hello i am your hot lil horny toy i am the one you dream about i am a very open minded person love to talk about and any subject fantasy is my way of life ultimate in sex play ummmmmmmmmmmmmm i am wet and ready for you it is not your looks but your imagination that matters most with my sexy voice i can make your dream come true hurry up call me let me cummmmm for you toll free teen for phone billing _______________________________________________ sign up for your own free personalized e mail at mail com url_token'

In [22]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def tokenize_and_lemmatize(text):
    doc = nlp(text)
    return [
        token.lemma_
        for token in doc
        if token.is_alpha or token.text in {"url_token", "email_token"}
    ]


In [23]:
base_path = "/content/drive/MyDrive/NLP1/model"

os.makedirs(base_path, exist_ok=True)

nlp.to_disk(base_path + "/spacy_pipeline")



In [24]:
df = df[df["clean_text"].str.len() < 50000]


In [25]:
df['tokens']=df['clean_text'].apply(tokenize_and_lemmatize)

In [26]:
df['tokens']

,tokens
0,"[re, disc, uniformitarianism, re, sex, lang, d..."
1,"[the, other, side, of, galicismos, galicismo, ..."
2,"[re, equistar, deal, ticket, be, you, still, a..."
3,"[hello, I, be, your, hot, lil, horny, toy, I, ..."
4,"[software, at, incredibly, low, price, low, dr..."
...,...
18644,"[rick, moen, a, crit, I, m, confused, I, think..."
18645,"[date, a, lonely, housewife, always, want, to,..."
18646,"[request, submit, access, request, for, anita,..."
18647,"[re, important, prc, mtg, hi, dorn, john, as, ..."


In [27]:
def get_tokens(tokens):
  cleaned_tokens = []
  for token in tokens:
    if token in NAMES: continue
    if token in STOPWORDS: continue
    if (len(token) == 1 and token.isalpha() and token not in {"a", "i"}):continue
    if pos_tag([token])[0][1][:2] not in ["NN", "VB", "JJ", "RB"] and token not in{"url_token","email_token"}: continue
    cleaned_tokens.append(token)

  return cleaned_tokens

In [28]:
df['tokens']=df['tokens'].apply(get_tokens)

In [29]:
df["text_length"] = df["clean_text"].str.len()
df.sort_values("text_length", ascending=False).head()


,Email Text,Email Type,language,clean_text,tokens,text_length
15915,enron mentions - 11 / 19 / 01 headed for a fal...,Safe Email,en,enron mentions headed for a fall companies iss...,"[enron, mention, head, fall, company, issue, s...",47572
1881,enron mentions enrononline competitor reports ...,Safe Email,en,enron mentions enrononline competitor reports ...,"[enron, mention, enrononline, competitor, repo...",47307
14274,\n---259765881.993893904265.JavaMail.RovAdmin....,Phishing Email,en,javamail rovadmin rovweb content type text pla...,"[javamail, rovadmin, rovweb, content, type, te...",46414
6073,enron mentions - 11 / 30 / 01 enron australia ...,Safe Email,en,enron mentions enron australia is barred from ...,"[enron, mention, enron, australia, bar, tradin...",46076
2099,enron mentions enron names special committee t...,Safe Email,en,enron mentions enron names special committee t...,"[enron, mention, enron, name, special, committ...",43071


In [30]:
df['tokens']

,tokens
0,"[disc, uniformitarianism, sex, lang, dick, hud..."
1,"[side, galicismos, galicismo, spanish, term, n..."
2,"[equistar, deal, ticket, still, available, ass..."
3,"[hello, hot, lil, horny, toy, dream, open, min..."
4,"[software, incredibly, low, price, low, draper..."
...,...
18644,"[rick, moen, crit, confused, think, gpl, ed, m..."
18645,"[date, lonely, housewife, always, want, date, ..."
18646,"[request, submit, access, request, anita, dupo..."
18647,"[important, prc, mtg, hi, dorn, john, discover..."


In [31]:
df.drop_duplicates(subset="clean_text", inplace=True)

In [32]:
full_vocabulary = []
vocabulary = {category: [] for category in df["Email Type"].unique()}
for _, row in df.iterrows():
  tokens = row["tokens"]
  sentiment = row["Email Type"]
  vocabulary[sentiment] += tokens
  full_vocabulary += tokens

In [33]:
print(f"Cantidad de tokens del vocabulario: {len(FreqDist(full_vocabulary))}")

Cantidad de tokens del vocabulario: 107014


In [34]:
FreqDist(full_vocabulary).most_common(500)

[('enron', 13977),
 ('ect', 10972),
 ('get', 10408),
 ('use', 9695),
 ('url_token', 9651),
 ('please', 9428),
 ('make', 9280),
 ('com', 9172),
 ('mail', 8706),
 ('information', 8642),
 ('company', 8433),
 ('new', 8401),
 ('email', 8184),
 ('time', 8183),
 ('list', 8150),
 ('language', 8133),
 ('send', 8120),
 ('work', 7362),
 ('say', 6414),
 ('address', 6233),
 ('report', 6175),
 ('university', 6166),
 ('know', 6125),
 ('need', 6036),
 ('business', 5920),
 ('subject', 5850),
 ('order', 5840),
 ('go', 5826),
 ('also', 5817),
 ('message', 5744),
 ('free', 5667),
 ('year', 5498),
 ('day', 5497),
 ('well', 5425),
 ('see', 5370),
 ('include', 5359),
 ('hou', 5280),
 ('people', 5136),
 ('price', 5111),
 ('receive', 5098),
 ('take', 5049),
 ('good', 5048),
 ('email_token', 5044),
 ('http', 4980),
 ('system', 4968),
 ('call', 4840),
 ('look', 4787),
 ('name', 4785),
 ('thank', 4763),
 ('want', 4593),
 ('program', 4466),
 ('service', 4426),
 ('money', 4409),
 ('find', 4328),
 ('number', 4237),


In [35]:
#Guardamos los archivos necesarios
#with open('/content/drive/MyDrive/NLP1/lemmatizer.pkl', 'wb') as f:
    #pickle.dump(lemmatizer,f)

In [36]:
df['tokens'].iloc[3]

['hello',
 'hot',
 'lil',
 'horny',
 'toy',
 'dream',
 'open',
 'minded',
 'person',
 'love',
 'talk',
 'subject',
 'fantasy',
 'way',
 'life',
 'ultimate',
 'sex',
 'play',
 'ummmmmmmmmmmmmm',
 'wet',
 'ready',
 'look',
 'imagination',
 'matter',
 'sexy',
 'voice',
 'make',
 'dream',
 'come',
 'true',
 'hurry',
 'call',
 'let',
 'cummmmm',
 'toll',
 'free',
 'teen',
 'phone',
 'billing',
 'sign',
 'free',
 'personalize',
 'mail',
 'mail',
 'com',
 'url_token']

In [37]:
most_common_tokens = set()
for category in df["Email Type"].unique():
  most_common_tokens = most_common_tokens.union(set([token for token, _ in FreqDist(vocabulary[category]).most_common(500)]))
most_common_tokens

{'able',
 'abstract',
 'ac',
 'accept',
 'access',
 'account',
 'acquisition',
 'act',
 'action',
 'actually',
 'ad',
 'add',
 'additional',
 'address',
 'adobe',
 'adult',
 'advertisement',
 'advertising',
 'advice',
 'age',
 'agent',
 'agree',
 'agreement',
 'allow',
 'already',
 'also',
 'always',
 'america',
 'american',
 'amount',
 'analysis',
 'analyst',
 'announce',
 'answer',
 'anticipate',
 'anyone',
 'anything',
 'appear',
 'application',
 'apply',
 'approach',
 'approve',
 'april',
 'area',
 'article',
 'ask',
 'aspect',
 'asset',
 'assistance',
 'associate',
 'attach',
 'august',
 'author',
 'available',
 'away',
 'back',
 'bank',
 'base',
 'become',
 'begin',
 'believe',
 'benefit',
 'big',
 'bill',
 'bit',
 'body',
 'bonus',
 'book',
 'box',
 'br',
 'brand',
 'break',
 'bring',
 'build',
 'bulk',
 'business',
 'buy',
 'california',
 'call',
 'canada',
 'capacity',
 'card',
 'case',
 'cash',
 'cause',
 'cc',
 'cd',
 'center',
 'chair',
 'change',
 'charge',
 'check',
 'chi

In [38]:
print(f"Cantidad de tokens comunes: {len(most_common_tokens)}")

Cantidad de tokens comunes: 695


In [39]:
# Guardar los archivos necesarios para preprocesar texto
with open("/content/drive/MyDrive/NLP1/most_common_tokens.pkl", "wb") as f:
  pickle.dump(most_common_tokens, f)

In [40]:
def get_tokens_training(email,most_common_tokens,simplifier_func=None):
  tokens = tokenize_and_lemmatize(email)
  cleaned_tokens = []
  for token in tokens:
    if not token.isalpha(): continue
    if token in NAMES: continue
    token = token.lower()
    if token in STOPWORDS: continue
    if pos_tag([token])[0][1][:2] not in ["NN", "VB", "JJ", "RB"] and token not in{"url_token","email_token"} : continue
    if simplifier_func: token = simplifier_func(token)
    if token not in most_common_tokens: continue
    cleaned_tokens.append(token)

  return " ".join(cleaned_tokens)

In [41]:
df['tokens training']=df['Email Text'].apply(lambda x:get_tokens_training(x,most_common_tokens))

#Vecorizacion

In [42]:
#Aquí desarrollamos el método Binary Term Frequency
vectorizer_binary = CountVectorizer(binary=True)
vectorizer_binary.fit(df["tokens training"])

CountVectorizer(binary=True)

In [43]:
X_binary = pd.DataFrame(
    vectorizer_binary.transform(df["tokens training"]).toarray(),
    columns=vectorizer_binary.get_feature_names_out(),
    index=df.index
)
X_binary

,able,abstract,ac,accept,access,account,acquisition,act,action,actually,...,workshop,world,worldwide,write,www,xp,year,yes,yet,york
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18644,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,1,0,0
18645,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
18646,0,0,0,0,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
18647,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [44]:
with open("/content/drive/MyDrive/NLP1/vectorizer_binary.pkl", "wb") as f:
  pickle.dump(vectorizer_binary, f)

In [45]:
#Aquí desarrollamos el método Bag of Words (BoW)
vectorizer_bow = CountVectorizer()
vectorizer_bow.fit(df["tokens training"])


CountVectorizer()

In [46]:
X_bow = pd.DataFrame(
    vectorizer_bow.transform(df["tokens training"]).toarray(),
    columns=vectorizer_bow.get_feature_names_out(),
    index=df.index
)
X_bow

,able,abstract,ac,accept,access,account,acquisition,act,action,actually,...,workshop,world,worldwide,write,www,xp,year,yes,yet,york
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18644,0,0,0,0,0,0,0,0,0,2,...,0,0,0,1,0,0,0,2,0,0
18645,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
18646,0,0,0,0,1,0,0,1,0,0,...,0,0,0,1,0,0,0,0,0,0
18647,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
with open("/content/drive/MyDrive/NLP1/vectorizer_bow.pkl", "wb") as f:
  pickle.dump(vectorizer_bow, f)

In [48]:
# Aquí desarrollamos el método más efectivo, el TF-IDF (Term Frequency-Inverse Document Frequency)
vectorizer_tfidf = TfidfVectorizer()
vectorizer_tfidf.fit(df["tokens training"])

TfidfVectorizer()

In [49]:
X_tf = pd.DataFrame(
    vectorizer_tfidf.transform(df["tokens training"]).toarray(),
    columns=vectorizer_tfidf.get_feature_names_out(),
    index=df.index
)
X_tf

,able,abstract,ac,accept,access,account,acquisition,act,action,actually,...,workshop,world,worldwide,write,www,xp,year,yes,yet,york
0,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.206089,0.0,0.0
1,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0
2,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0
3,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0
4,0.18517,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.138945,0.00000,0.0,0.0,0.211332,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18644,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.122841,...,0.0,0.0,0.0,0.041395,0.00000,0.0,0.0,0.125922,0.0,0.0
18645,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.25395,0.0,0.0,0.000000,0.0,0.0
18646,0.00000,0.0,0.0,0.0,0.128267,0.0,0.0,0.133399,0.0,0.000000,...,0.0,0.0,0.0,0.094381,0.00000,0.0,0.0,0.000000,0.0,0.0
18647,0.00000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,...,0.0,0.0,0.0,0.000000,0.00000,0.0,0.0,0.000000,0.0,0.0


In [50]:
with open("/content/drive/MyDrive/NLP1/vectorizer_tfidf.pkl", "wb") as f:
  pickle.dump(vectorizer_tfidf, f)

In [60]:
Y=df['Email Type']
X_text=df['tokens training']

In [55]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizadores = {
    "BoW": CountVectorizer(),
    "Binary": CountVectorizer(binary=True),
    "TF-IDF": TfidfVectorizer()
}




In [56]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB

modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "MultinomialNB": MultinomialNB()
}


In [57]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


f1_phishing = make_scorer(
    f1_score,
    pos_label="Phishing Email"
)


In [52]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [61]:
results = []

for vec_name, vectorizer in vectorizadores.items():

    for model_name, model in modelos.items():

        pipeline = Pipeline([
            ("vectorizer", vectorizer),
            ("classifier", model)
        ])

        scores = cross_val_score(
            pipeline,
            X_text,
            Y,
            cv=cv,
            scoring=f1_phishing
        )

        results.append({
            "Vectorizacion": vec_name,
            "Modelo": model_name,
            "CV F1 Mean": scores.mean(),
            "CV F1 Std": scores.std()
        })

results_df = pd.DataFrame(results).sort_values(
    by="CV F1 Mean",
    ascending=False
)

print(results_df)


  Vectorizacion               Modelo  CV F1 Mean  CV F1 Std
7        TF-IDF        Random Forest    0.945636   0.002639
4        Binary        Random Forest    0.945554   0.004042
1           BoW        Random Forest    0.945479   0.003361
6        TF-IDF  Logistic Regression    0.942385   0.003211
3        Binary  Logistic Regression    0.941744   0.005059
0           BoW  Logistic Regression    0.936258   0.005936
8        TF-IDF        MultinomialNB    0.924409   0.002064
5        Binary        MultinomialNB    0.917814   0.004418
2           BoW        MultinomialNB    0.904000   0.002585


In [64]:
best_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    Y,
    test_size=0.2,
    stratify=Y,
    random_state=42
)

best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)


In [65]:
best_pipeline.fit(X_train, y_train)

# Predicciones en train
y_train_pred = best_pipeline.predict(X_train)

# Predicciones en test
y_test_pred = best_pipeline.predict(X_test)

from sklearn.metrics import classification_report

print("===== TRAIN =====")
print(classification_report(y_train, y_train_pred))

print("===== TEST =====")
print(classification_report(y_test, y_test_pred))


===== TRAIN =====
                precision    recall  f1-score   support

Phishing Email       0.93      0.97      0.95      4994
    Safe Email       0.98      0.96      0.97      8426

      accuracy                           0.96     13420
     macro avg       0.96      0.96      0.96     13420
  weighted avg       0.96      0.96      0.96     13420

===== TEST =====
                precision    recall  f1-score   support

Phishing Email       0.93      0.96      0.95      1248
    Safe Email       0.98      0.96      0.97      2107

      accuracy                           0.96      3355
     macro avg       0.95      0.96      0.96      3355
  weighted avg       0.96      0.96      0.96      3355



In [54]:
df['Email Type'].value_counts()

,count
Email Type,
Safe Email,10533
Phishing Email,6242


In [68]:
import joblib

best_pipeline.fit(X_train, y_train)

joblib.dump(best_pipeline, "/content/drive/MyDrive/NLP1/phishing_model.pkl")


['/content/drive/MyDrive/NLP1/phishing_model.pkl']

In [69]:
import os
os.getcwd()


'/content'

In [70]:
MODEL_PATH = "/content/drive/MyDrive/NLP1/phishing_model.pkl"

joblib.dump(best_pipeline, MODEL_PATH)


['/content/drive/MyDrive/NLP1/phishing_model.pkl']